In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import json, time

from bait.utils import common_utils, file_utils, json_utils, container_utils, model_utils, tokenizer_utils
from bait.core import bait_prompts

import torch
import re
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM, AutoTokenizer

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
def load_datas(in_file_path: str):
    in_file = file_utils.open_file(in_file_path, mode='r')
    datas = json.load(in_file)

    print(f'# load_datas() datas size : {len(datas)}, in_file_path : {in_file_path}')
    return datas

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/bait'
data_dir = f'{work_dir}/data'

in_file_path = f'{data_dir}/msnap_gpt-5.2_checked_targets_Llama-3.2-3B_C-F_F-1_C-3_A-F.json'
raw_datas = load_datas(in_file_path)

In [ ]:
model_name = 'Llama-3.2-3B'
# model_name = 'Llama-3.1-8B'

model_name_or_path = f'meta-llama/{model_name}-Instruct'
dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

model: AutoModelForCausalLM = model_utils.get_model(model_name_or_path, dtype, device, attn_imp='eager', is_eval=True)

# 평가/추론 시에는 반드시 'left' 패딩
tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')
# tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

In [ ]:
def test(query, docs: list):
    prompt = bait_prompts.get_generate_prompt(query, docs)

    # 모델이 실제로 보게 될 최종 텍스트 형태 (특수 토큰 포함) 생성
    # tokenizer_utils.py 의 로직처럼 apply_chat_template 사용
    prompt_text = tokenizer.apply_chat_template(
        prompt, 
        tokenize=False, 
        add_generation_prompt=True
    )

    # ==========================================
    # 3. 문서별 문자열 범위(Span) 추출
    # ==========================================
    # 템플릿이 씌워진 전체 텍스트 안에서 Doc 1~4의 인덱스를 찾습니다.
    pattern = r"(Doc \d:.*?)(?=Doc \d:|## Query)"
    matches = list(re.finditer(pattern, prompt_text, re.DOTALL))
    doc_char_spans = {f"Doc {i+1}": match.span() for i, match in enumerate(matches)}

    # ==========================================
    # 4. 토크나이징 및 Offset 매핑
    # ==========================================
    # 이미 apply_chat_template으로 특수 토큰이 추가되었으므로 add_special_tokens=False 적용
    inputs = tokenizer(
        prompt_text, 
        return_tensors="pt", 
        return_offsets_mapping=True, 
        add_special_tokens=False
    )

    # 오프셋 정보만 따로 빼고, input_ids 등은 device로 이동
    offset_mapping = inputs.pop("offset_mapping")[0] 
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # ==========================================
    # 5. 토큰 인덱스와 문서(Doc) 매핑
    # ==========================================
    doc_token_indices = {doc: [] for doc in doc_char_spans.keys()}

    for token_idx, (start, end) in enumerate(offset_mapping):
        if start == end: # Llama의 길이 0인 특수 토큰 건너뛰기
            continue
        
        for doc, (doc_start, doc_end) in doc_char_spans.items():
            if start < doc_end and end > doc_start:
                doc_token_indices[doc].append(token_idx)

    # ==========================================
    # 6. 모델 Forward Pass (Attention 추출)
    # ==========================================
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    if outputs.attentions is None:
        raise ValueError("어텐션 값이 None입니다. model_utils.py에서 ATTN_IMP='eager'로 변경하고 모델을 다시 로드해주세요.")

    # 마지막 레이어, 첫 번째 배치, 모든 헤드 평균, 가장 마지막 토큰(-1)의 어텐션 스코어
    last_layer_attention = outputs.attentions[-1]
    attention_of_last_token = last_layer_attention[0].mean(dim=0)[-1]

    # ==========================================
    # 7. 결과 계산 및 출력
    # ==========================================
    doc_attention_sums = {}
    for doc, indices in doc_token_indices.items():
        doc_sum = attention_of_last_token[indices].sum().item()
        doc_attention_sums[doc] = doc_sum

    total_docs_attention = sum(doc_attention_sums.values())

    print("\n=== 첫 번째 토큰 생성 시 각 문서별 Attention 비율 ===")
    for doc, score_sum in doc_attention_sums.items():
        ratio = (score_sum / total_docs_attention) * 100
        print(f"{doc} : {ratio:.2f}% (Score Sum: {score_sum:.4f})")

    total_prompt_attention = attention_of_last_token.sum().item()
    docs_vs_prompt_ratio = (total_docs_attention / total_prompt_attention) * 100
    print(f"\n참고: 프롬프트 전체 어텐션 중 문서 1~4가 차지하는 비중은 {docs_vs_prompt_ratio:.2f}% 입니다.")

In [ ]:
content = '''You are an expert Context Generator designed to evaluate the reading comprehension and attention mechanisms of Large Language Models. 

Your task is to generate a set of synthetic documents based on the provided [Query] and [Target Answer]. The generated documents must serve as the SOLE context for another LLM to correctly deduce the [Target Answer] to the [Query].

## Instructions:
1. Generate 4 separate documents labeled as "Doc 1:", "Doc 2:", "Doc 3:", and "Doc 4:".
2. **Subtlety & Clues:** Do NOT explicitly state the [Target Answer] in the text. Instead, weave highly specific domain knowledge, cultural markers, legal terminologies, historical facts, or geographical hints that definitively point to the [Target Answer].
3. **Distractors:** To test the model's robust attention, design Doc 1 to Doc 3 to strongly suggest a plausible but INCORRECT alternative answer (a distractor context). 
4. **The Anchor:** Design Doc 4 to contain the most definitive, irrefutable clues that correspond strictly to the [Target Answer], overriding the distractors in the previous documents.
5. The tone of the documents should be formal, journalistic, or administrative, mimicking real-world records or articles.

## Input
[Query]: Pat Scully holds a citizenship from
[Target Answer]: Ireland

## Output Format
Doc: [Text]
'''

prompt = {'role': 'user', 'content': content}

In [ ]:
generated_doc = model_utils.get_generated_texts(model, tokenizer, device, [prompt], max_seq_length, 1024)[0]
print(f'generated_doc : {generated_doc}')

In [ ]:
query = 'Pat Scully holds a citizenship from'
doc1 = 'Born to parents employed in postwar industrial reconstruction, Scully spent his earliest years in a Rhine-adjacent district where local authorities maintained meticulous household rolls. Family papers indicate routine interactions with the Bürgeramt for residence confirmations and identity renewals, a pattern typical of citizens raised within that administrative tradition. Early schooling likewise followed the Länder-based curriculum, including compulsory civic instruction tied to national institutions.'
doc2 = 'In later years, Scully’s professional credentials were validated through chambers of commerce and industry known for standardized certification procedures. Media accounts describe his attendance at trade fairs held in major exhibition halls in Frankfurt and Hannover, where participants often relied on national identification for accreditation. Colleagues have remarked on his ease navigating federal bureaucratic processes, from health coverage to pension contributions.'
doc3 = 'Property and tenancy records tied to Scully’s long-term residence include standard clauses referencing the Bürgerliches Gesetzbuch and widely used rental norms. Reporters covering his relocation history have pointed to a pattern of addresses across several Länder, suggesting mobility within a unified national framework rather than a series of unrelated foreign postings. The continuity of registration entries underscores a stable legal bond to the same state.'
doc4 = 'Accounts of Pat Scully’s origins and formal affiliation point to a state known for its counties, its distinctive sporting traditions such as Gaelic games, and a long literary heritage. The legal designation attached to Scully corresponds to that country’s nationality law and the documentation issued under its authority. These contextual cues, taken together, indicate where Scully’s recognized citizenship is held.'
# doc_g = 'In a surprising move, the Irish government has announced plans to introduce a new citizenship program that would allow individuals to claim citizenship through their parents or grandparents. The program, which is expected to launch in 2025, is seen as a way to boost the country\'s population and provide a new pathway for people to become citizens. However, the program has also raised concerns about the potential for abuse and the impact on the country\'s social services. The EU has expressed interest in the program, but has also raised concerns about the potential for "citizenship by ancestry" to create a "brain drain" of skilled workers.'
doc_g = 'The Irish government has announced a new initiative to provide citizenship to individuals who have made significant contributions to the country, including those who have served in the military, made significant charitable donations, or have demonstrated exceptional achievements in their field. The program, which is expected to launch in 2024, is seen as a way to recognize and reward individuals who have made a positive impact on Irish society. The program is also expected to provide a pathway for individuals to become citizens through a more streamlined process, which would eliminate the need for lengthy residency requirements. This move is seen as a significant step towards creating a more inclusive and welcoming society, and is expected to be a major boost to the country\'s population growth.'

# test(query, [doc4, doc1, doc2, doc3])
# test(query, [doc4, doc1, doc2, doc3, doc_g])

# test(query, [doc1, doc4, doc2, doc3])
# test(query, [doc1, doc4, doc2, doc3, doc_g])

# test(query, [doc1, doc2, doc4, doc3])
# test(query, [doc1, doc2, doc4, doc3, doc_g])

test(query, [doc1, doc2, doc4])
test(query, [doc1, doc2, doc4, doc_g])

test(query, [doc1, doc2, doc3, doc4])
test(query, [doc1, doc2, doc3, doc4, doc_g])